ICD-10 coding pipeline — evaluation across all pipeline steps
*Co-authored with CoCo*

# 04: Pipeline Evaluation

Evaluates all three pipeline steps against `CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH`.

**Metrics computed:**
- **Step 1 (Extraction):** LLM-as-judge recall — was each GT diagnosis captured in extracted findings?
- **Step 2 (Search):** Search recall — does the correct ICD-10 code appear in the candidate set?
- **Step 3 (Matching):** End-to-end accuracy — was the correct code assigned by the pipeline?
- **Funnel:** Accuracy at each step (what % pass through)
- **Verification status:** Per GT code breakdown across all 2,112 assignments

In [ ]:
%%sql -r ctx
-- Context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICD10_CODING_APP;
USE WAREHOUSE COMPUTE_WH;
ALTER SESSION SET QUERY_TAG = 'icd10_v2:evaluation';

---
## Eval 1: Extraction Recall (LLM-as-Judge)

For each GT code, ask the LLM: does any extracted finding correspond to this diagnosis?

In [ ]:
%%sql -r eval_config
SET EVAL_MODEL = 'claude-4-sonnet';

In [ ]:
%%sql -r eval1
-- Extraction eval: LLM-as-judge for each distinct GT (chase, code) pair
CREATE OR REPLACE TABLE EXPERIMENTS.EVAL_EXTRACTION AS
WITH hcc_deduped AS (
    SELECT ICD_CODE, DESCRIPTION, HCC_V28
    FROM CHART_REVIEW_DB.RAW.ICD10_HCC_MAPPINGS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ICD_CODE ORDER BY RA_2026_V28 DESC) = 1
),
gt AS (
    SELECT DISTINCT
        gt.CHASE_ID,
        gt.DIAGNOSIS_CODE AS GT_CODE,
        m.DESCRIPTION AS GT_DESCRIPTION,
        m.HCC_V28 AS GT_HCC_V28
    FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH gt
    LEFT JOIN hcc_deduped m
        ON UPPER(REPLACE(gt.DIAGNOSIS_CODE, '.', '')) = UPPER(REPLACE(m.ICD_CODE, '.', ''))
    WHERE gt.ICD_CODE_DISPOSITION = 'ADD'
),
findings_per_chase AS (
    SELECT
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        LISTAGG(
            FINDING_SEQ || '. ' || FINDING
            || ' [' || CATEGORY || ']'
            || COALESCE(' (acuity: ' || ACUITY || ')', '')
            || COALESCE(' (laterality: ' || LATERALITY || ')', '')
            || COALESCE(' (body_site: ' || BODY_SITE || ')', '')
            || COALESCE(' (causal_link: ' || CAUSAL_LINK || ')', '')
            || COALESCE(' (severity: ' || SEVERITY || ')', ''),
            '\n'
        ) WITHIN GROUP (ORDER BY FINDING_SEQ) AS ALL_FINDINGS_TEXT
    FROM PROCESSING.ENCOUNTER_FINDINGS
    GROUP BY 1
)
SELECT
    g.CHASE_ID, g.GT_CODE, g.GT_DESCRIPTION, g.GT_HCC_V28,
    SNOWFLAKE.CORTEX.COMPLETE(
        $EVAL_MODEL,
        CONCAT(
            'You are evaluating a clinical NLP extraction system. ',
            'Given extracted findings (with metadata: category, acuity, laterality, body_site, causal_link, severity), ',
            'determine if ANY finding corresponds to the target diagnosis.', CHR(10), CHR(10),
            'Target: ', g.GT_CODE, ' - ', COALESCE(g.GT_DESCRIPTION, 'Unknown'), CHR(10), CHR(10),
            'Extracted Findings:', CHR(10),
            LEFT(COALESCE(f.ALL_FINDINGS_TEXT, '(none)'), 10000), CHR(10), CHR(10),
            'Respond ONLY: {"verdict": true/false, "rationale": "<one sentence>"}'
        )
    ) AS JUDGE_RAW,
    TRY_PARSE_JSON(REGEXP_SUBSTR(JUDGE_RAW, '\\{.*\\}', 1, 1, 's')):verdict::BOOLEAN AS VERDICT,
    TRY_PARSE_JSON(REGEXP_SUBSTR(JUDGE_RAW, '\\{.*\\}', 1, 1, 's')):rationale::VARCHAR AS RATIONALE
FROM gt g
INNER JOIN findings_per_chase f ON g.CHASE_ID = f.CHASE_ID;

In [ ]:
%%sql -r eval1_summary
-- Extraction recall summary
SELECT
    COUNT(*) AS TOTAL_GT_CODES,
    COUNT_IF(VERDICT = TRUE) AS CAPTURED,
    COUNT_IF(VERDICT = FALSE) AS MISSED,
    ROUND(COUNT_IF(VERDICT = TRUE) * 100.0 / COUNT(*), 2) AS EXTRACTION_RECALL_PCT
FROM EXPERIMENTS.EVAL_EXTRACTION;

---
## Eval 2: Search Recall

For each GT code that was extracted, does it appear in the search candidates?

In [ ]:
%%sql -r eval2
-- Search eval: correct code in candidates?
-- Uses only codes that passed extraction (one row per chase+code)
CREATE OR REPLACE TABLE EXPERIMENTS.EVAL_SEARCH AS
WITH gt_extracted AS (
    SELECT CHASE_ID, GT_CODE, GT_DESCRIPTION, GT_HCC_V28
    FROM EXPERIMENTS.EVAL_EXTRACTION
    WHERE VERDICT = TRUE
    QUALIFY ROW_NUMBER() OVER (PARTITION BY CHASE_ID, GT_CODE ORDER BY GT_CODE) = 1
),
search_candidates AS (
    SELECT
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        UPPER(REPLACE(ICD10_CODE, '.', '')) AS CANDIDATE_NODOT,
        MIN(FINAL_RANK) AS BEST_RANK
    FROM MATCHING.CANDIDATES_MERGED
    GROUP BY 1, 2
)
SELECT
    g.CHASE_ID, g.GT_CODE, g.GT_DESCRIPTION, g.GT_HCC_V28,
    CASE WHEN sc.CANDIDATE_NODOT IS NOT NULL THEN TRUE ELSE FALSE END AS FOUND_IN_SEARCH,
    sc.BEST_RANK
FROM gt_extracted g
LEFT JOIN search_candidates sc
    ON sc.CHASE_ID = g.CHASE_ID
    AND sc.CANDIDATE_NODOT = UPPER(REPLACE(g.GT_CODE, '.', ''));

In [ ]:
%%sql -r eval2_summary
-- Search recall summary
SELECT
    COUNT(*) AS TOTAL_EXTRACTED_GT,
    COUNT_IF(FOUND_IN_SEARCH) AS FOUND,
    COUNT_IF(NOT FOUND_IN_SEARCH) AS NOT_FOUND,
    ROUND(COUNT_IF(FOUND_IN_SEARCH) * 100.0 / COUNT(*), 2) AS SEARCH_RECALL_PCT,
    ROUND(AVG(CASE WHEN FOUND_IN_SEARCH THEN BEST_RANK END), 2) AS AVG_RANK_WHEN_FOUND
FROM EXPERIMENTS.EVAL_SEARCH;

---
## Eval 3: End-to-End Matching

For all GT codes, was the correct code assigned by the pipeline?

In [ ]:
%%sql -r eval_detail
-- Full detail table: every GT assignment with verification across all 3 steps
-- Produces exactly 2,112 rows (one per GT row)
CREATE OR REPLACE TABLE EXPERIMENTS.EVAL_FULL_DETAIL AS
WITH hcc_deduped AS (
    SELECT ICD_CODE, DESCRIPTION, HCC_V28
    FROM CHART_REVIEW_DB.RAW.ICD10_HCC_MAPPINGS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ICD_CODE ORDER BY RA_2026_V28 DESC) = 1
),
gt AS (
    SELECT
        gt.CHASE_ID,
        gt.DIAGNOSIS_CODE AS GT_CODE,
        m.DESCRIPTION AS GT_DESCRIPTION,
        m.HCC_V28 AS GT_HCC,
        gt.DOS_START_DATE
    FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH gt
    LEFT JOIN hcc_deduped m
        ON UPPER(REPLACE(gt.DIAGNOSIS_CODE, '.', '')) = UPPER(REPLACE(m.ICD_CODE, '.', ''))
    WHERE gt.ICD_CODE_DISPOSITION = 'ADD'
),
eval_extraction_deduped AS (
    SELECT CHASE_ID, GT_CODE, VERDICT
    FROM EXPERIMENTS.EVAL_EXTRACTION
    QUALIFY ROW_NUMBER() OVER (PARTITION BY CHASE_ID, GT_CODE ORDER BY VERDICT DESC) = 1
),
eval_search_deduped AS (
    SELECT CHASE_ID, GT_CODE, FOUND_IN_SEARCH, BEST_RANK
    FROM EXPERIMENTS.EVAL_SEARCH
    QUALIFY ROW_NUMBER() OVER (PARTITION BY CHASE_ID, GT_CODE ORDER BY BEST_RANK ASC NULLS LAST) = 1
),
pipeline_predictions AS (
    SELECT CHASE_ID, ASSIGNED_NODOT,
        MAX_BY(ASSIGNED_CODE, CONFIDENCE) AS PREDICTED_CODE,
        MAX_BY(ASSIGNED_DESCRIPTION, CONFIDENCE) AS PREDICTED_DESCRIPTION,
        MAX(CONFIDENCE) AS CONFIDENCE
    FROM (
        SELECT
            UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
            UPPER(REPLACE(ASSIGNED_CODE, '.', '')) AS ASSIGNED_NODOT,
            ASSIGNED_CODE,
            ASSIGNED_DESCRIPTION,
            CONFIDENCE
        FROM MATCHING.FINAL_ASSIGNMENTS_V2
        WHERE ASSIGNED_CODE IS NOT NULL AND ASSIGNMENT_RAW NOT ILIKE '%NONE%'
    )
    GROUP BY CHASE_ID, ASSIGNED_NODOT
)
SELECT
    g.CHASE_ID, g.GT_CODE, g.GT_DESCRIPTION, g.GT_HCC, g.DOS_START_DATE,
    e.VERDICT AS STEP1_EXTRACTED,
    s.FOUND_IN_SEARCH AS STEP2_IN_CANDIDATES,
    s.BEST_RANK AS STEP2_RANK,
    CASE WHEN p.ASSIGNED_NODOT IS NOT NULL THEN TRUE ELSE FALSE END AS STEP3_MATCHED,
    p.PREDICTED_CODE, p.PREDICTED_DESCRIPTION, p.CONFIDENCE AS STEP3_CONFIDENCE,
    CASE
        WHEN p.ASSIGNED_NODOT IS NOT NULL AND p.CONFIDENCE >= 0.75 THEN 'CORRECT_HIGH_CONF'
        WHEN p.ASSIGNED_NODOT IS NOT NULL AND p.CONFIDENCE < 0.75 THEN 'CORRECT_LOW_CONF'
        WHEN s.FOUND_IN_SEARCH = TRUE THEN 'IN_CANDIDATES_NOT_MATCHED'
        WHEN COALESCE(e.VERDICT, FALSE) = TRUE THEN 'EXTRACTED_NOT_IN_CANDIDATES'
        ELSE 'NOT_EXTRACTED'
    END AS VERIFICATION_STATUS
FROM gt g
LEFT JOIN eval_extraction_deduped e ON e.CHASE_ID = g.CHASE_ID AND e.GT_CODE = g.GT_CODE
LEFT JOIN eval_search_deduped s ON s.CHASE_ID = g.CHASE_ID AND UPPER(REPLACE(s.GT_CODE, '.', '')) = UPPER(REPLACE(g.GT_CODE, '.', ''))
LEFT JOIN pipeline_predictions p ON p.CHASE_ID = g.CHASE_ID AND p.ASSIGNED_NODOT = UPPER(REPLACE(g.GT_CODE, '.', ''));

---
## Results

In [ ]:
%%sql -r status_counts
-- Verification status counts (all 2,112 GT assignments)
SELECT
    VERIFICATION_STATUS,
    COUNT(*) AS COUNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS PCT
FROM EXPERIMENTS.EVAL_FULL_DETAIL
GROUP BY VERIFICATION_STATUS
ORDER BY COUNT DESC;

In [ ]:
%%sql -r funnel
-- Pipeline funnel: accuracy at each step (DISTINCT codes + ALL assignments)
WITH gt_distinct AS (
    SELECT DISTINCT CHASE_ID, UPPER(REPLACE(GT_CODE, '.', '')) AS GT_NODOT
    FROM EXPERIMENTS.EVAL_FULL_DETAIL
),
gt_all AS (
    SELECT CHASE_ID, UPPER(REPLACE(GT_CODE, '.', '')) AS GT_NODOT
    FROM EXPERIMENTS.EVAL_FULL_DETAIL
)
SELECT 'DISTINCT' AS GRAIN, 'Step 1: Extraction' AS STEP,
    COUNT(*) AS TOTAL, COUNT_IF(STEP1_EXTRACTED) AS PASSED,
    ROUND(COUNT_IF(STEP1_EXTRACTED) * 100.0 / COUNT(*), 2) AS ACCURACY_PCT
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, STEP1_EXTRACTED FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'DISTINCT', 'Step 2: Search',
    COUNT_IF(STEP1_EXTRACTED), COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES),
    ROUND(COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES) * 100.0 / NULLIF(COUNT_IF(STEP1_EXTRACTED), 0), 2)
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, STEP1_EXTRACTED, STEP2_IN_CANDIDATES FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'DISTINCT', 'Step 3: Matching',
    COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES), COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES AND STEP3_MATCHED),
    ROUND(COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES AND STEP3_MATCHED) * 100.0 / NULLIF(COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES), 0), 2)
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, STEP1_EXTRACTED, STEP2_IN_CANDIDATES, STEP3_MATCHED FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'DISTINCT', 'End-to-End',
    COUNT(*), COUNT_IF(STEP3_MATCHED),
    ROUND(COUNT_IF(STEP3_MATCHED) * 100.0 / COUNT(*), 2)
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, STEP3_MATCHED FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'Step 1: Extraction',
    COUNT(*), COUNT_IF(STEP1_EXTRACTED), ROUND(COUNT_IF(STEP1_EXTRACTED) * 100.0 / COUNT(*), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'Step 2: Search',
    COUNT_IF(STEP1_EXTRACTED), COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES),
    ROUND(COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES) * 100.0 / NULLIF(COUNT_IF(STEP1_EXTRACTED), 0), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'Step 3: Matching',
    COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES), COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES AND STEP3_MATCHED),
    ROUND(COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES AND STEP3_MATCHED) * 100.0 / NULLIF(COUNT_IF(STEP1_EXTRACTED AND STEP2_IN_CANDIDATES), 0), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'End-to-End',
    COUNT(*), COUNT_IF(STEP3_MATCHED), ROUND(COUNT_IF(STEP3_MATCHED) * 100.0 / COUNT(*), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL


ORDER BY GRAIN, STEP;

In [ ]:
%%sql -r top_missed
-- Top missed codes per step
SELECT GT_CODE, GT_DESCRIPTION, GT_HCC, VERIFICATION_STATUS,
    COUNT(*) AS TIMES_MISSED
FROM EXPERIMENTS.EVAL_FULL_DETAIL
WHERE VERIFICATION_STATUS != 'CORRECT_HIGH_CONF'
GROUP BY GT_CODE, GT_DESCRIPTION, GT_HCC, VERIFICATION_STATUS
ORDER BY TIMES_MISSED DESC
LIMIT 25;

---
## Experiment Logging

In [ ]:
%%sql -r log_exp
-- Log run
INSERT INTO EXPERIMENTS.RUNS (RUN_ID, STRATEGY, MODEL, PROMPT_VERSION, PARAMETERS_JSON, CREATED_AT)
SELECT
    'eval_v2_' || TO_CHAR(CURRENT_TIMESTAMP(), 'YYYYMMDD_HH24MISS'),
    'FULL_PIPELINE_V2',
    $EVAL_MODEL,
    'v2_modular',
    OBJECT_CONSTRUCT(
        'source', 'CHART_REVIEW_DB.RAW.ENCOUNTERS',
        'ground_truth', 'CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH',
        'extraction_recall', (SELECT ROUND(COUNT_IF(VERDICT = TRUE) * 100.0 / COUNT(*), 2) FROM EXPERIMENTS.EVAL_EXTRACTION),
        'search_recall', (SELECT ROUND(COUNT_IF(FOUND_IN_SEARCH) * 100.0 / COUNT(*), 2) FROM EXPERIMENTS.EVAL_SEARCH),
        'end_to_end_pct', (SELECT ROUND(COUNT_IF(STEP3_MATCHED) * 100.0 / COUNT(*), 2) FROM (SELECT DISTINCT CHASE_ID, GT_CODE, STEP3_MATCHED FROM EXPERIMENTS.EVAL_FULL_DETAIL))
    ),
    CURRENT_TIMESTAMP();

-- Log metrics
INSERT INTO EXPERIMENTS.METRICS (RUN_ID, METRIC_NAME, METRIC_VALUE, CREATED_AT)
WITH run AS (SELECT MAX(RUN_ID) AS RUN_ID FROM EXPERIMENTS.RUNS WHERE STRATEGY = 'FULL_PIPELINE_V2')
SELECT r.RUN_ID, 'extraction_recall',
    (SELECT COUNT_IF(VERDICT = TRUE) * 1.0 / COUNT(*) FROM EXPERIMENTS.EVAL_EXTRACTION), CURRENT_TIMESTAMP() FROM run r
UNION ALL
SELECT r.RUN_ID, 'search_recall',
    (SELECT COUNT_IF(FOUND_IN_SEARCH) * 1.0 / COUNT(*) FROM EXPERIMENTS.EVAL_SEARCH), CURRENT_TIMESTAMP() FROM run r
UNION ALL
SELECT r.RUN_ID, 'end_to_end_recall',
    (SELECT COUNT_IF(STEP3_MATCHED) * 1.0 / COUNT(*) FROM (SELECT DISTINCT CHASE_ID, GT_CODE, STEP3_MATCHED FROM EXPERIMENTS.EVAL_FULL_DETAIL)), CURRENT_TIMESTAMP() FROM run r
UNION ALL
SELECT r.RUN_ID, 'correct_high_conf_pct',
    (SELECT COUNT_IF(VERIFICATION_STATUS = 'CORRECT_HIGH_CONF') * 1.0 / COUNT(*) FROM EXPERIMENTS.EVAL_FULL_DETAIL), CURRENT_TIMESTAMP() FROM run r;

---
## Done

Compare runs in `ICD10_CODING_APP.EXPERIMENTS.METRICS` or Snowsight **AI & ML → Experiments**.